# Donation Impact Calculator

**Business goal:** Compute cost-per-impact-unit ratios from historical data so the
donate page can instantly show donors what their contribution achieves — e.g.
"Could provide 3 months of safe shelter for a girl".

## How It Works (the "Zestimate" analogy)

Just as Zillow pre-computes home value estimates from historical sales data so buyers
see instant predictions, we pre-compute **impact-per-peso ratios** from historical
donation and operational data. The ratios are stored in the database and served to the
frontend — no real-time computation needed as the donor types.

## Pipeline Structure

| Section | Purpose |
|---------|---------|
| 1–4 | **Descriptive pipeline** — load data, compute allocations, denominators, and ratios |
| 4b | **Ratio integrity validation** — completeness, plausibility bounds, temporal stability |
| 5 | **Explanatory OLS model** — what drives cost-per-unit variation across safehouses and time? |
| 6 | **Predictive model** — can we accurately forecast impact from a donation amount? (rolling backtest) |
| 7 | **Summary of relationships & causal claims** |
| 8–9 | Sanity check and DB write |

## Success Criteria

- Temporal coefficient of variation (CV) < 0.3 for displayed ratios
- OLS diagnostics pass (normality, homoscedasticity)
- Predictive MAE lower than naive baseline
- All ratios within domain-plausible bounds for Philippine nonprofit operations

**Output table:** `ml_donation_impact_rates` — follows `IsCurrent` / `ScoredAt` / `ModelVersion` pattern.

In [ ]:
import pandas as pd
import numpy as np
import datetime as _dt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from db_config import engine, USE_DB, text

print(f"USE_DB = {USE_DB}")

In [ ]:
CSV_DIR = Path('lighthouse_csv_v7')
if not CSV_DIR.is_dir():
    CSV_DIR = Path('ml-pipelines') / 'lighthouse_csv_v7'
assert CSV_DIR.is_dir(), f"CSV directory not found. Run from ml-pipelines/ directory."

def load_table(table_name: str) -> pd.DataFrame:
    """Load a table from Azure SQL when USE_DB=True, otherwise fall back to CSV."""
    if USE_DB:
        df = pd.read_sql_table(table_name, engine)
        print(f"  {table_name}: {len(df):,} rows (from DB)")
        return df
    csv_path = CSV_DIR / f"{table_name}.csv"
    if not csv_path.exists():
        print(f"  {table_name}: CSV not found at {csv_path} — returning empty DataFrame")
        return pd.DataFrame()
    df = pd.read_csv(csv_path)
    print(f"  {table_name}: {len(df):,} rows (from CSV)")
    return df

print("Loading tables...")
donations     = load_table('donations')
allocations   = load_table('donation_allocations')
metrics       = load_table('safehouse_monthly_metrics')
recordings    = load_table('process_recordings')
education     = load_table('education_records')
health        = load_table('health_wellbeing_records')
visitations   = load_table('home_visitations')
residents     = load_table('residents')
safehouses    = load_table('safehouses')

## 2. Prepare Donation Allocations by Program Area

In [ ]:
# Parse amount_allocated to numeric (it may be stored as string)
allocations['amount_allocated'] = pd.to_numeric(allocations['amount_allocated'], errors='coerce')
allocations = allocations.dropna(subset=['amount_allocated', 'program_area'])

# Parse allocation_date for temporal analysis later
allocations['allocation_date'] = pd.to_datetime(allocations['allocation_date'], errors='coerce')

total_by_area = allocations.groupby('program_area')['amount_allocated'].sum()
print("\nTotal allocated by program area (PHP):")
print(total_by_area.to_string())
print(f"\nGrand total allocated: ₱{total_by_area.sum():,.2f}")

## 3. Compute Denominators (Units of Impact)

In [ ]:
# Shelter: total bed-months from safehouse monthly metrics
metrics['active_residents'] = pd.to_numeric(metrics.get('active_residents', pd.Series(dtype='float64')), errors='coerce').fillna(0)
total_bed_months = metrics['active_residents'].sum()

# Counseling: total sessions
total_sessions = len(recordings)

# Education: total monthly records (each row = one girl's monthly record)
total_edu_months = len(education)

# Home visitations
total_visitations = len(visitations)

# Health: total monthly records
total_health_months = len(health)

# Residents served (for transport/general)
total_residents = len(residents)

print("\nUnits of impact (denominators):")
print(f"  Bed-months (shelter):     {total_bed_months:,.0f}")
print(f"  Counseling sessions:      {total_sessions:,}")
print(f"  Education record-months:  {total_edu_months:,}")
print(f"  Home visitations:         {total_visitations:,}")
print(f"  Health record-months:     {total_health_months:,}")
print(f"  Residents served:         {total_residents:,}")

## 4. Compute Cost-per-Unit Ratios

In [ ]:
# Map each program_area to its impact denominator and donor-friendly label
PROGRAM_MAPPING = {
    'Operations':  {'units': total_bed_months,   'label': 'month of safe shelter for a girl'},
    'Wellbeing':   {'units': total_sessions,     'label': 'counseling session'},
    'Education':   {'units': total_edu_months,   'label': 'month of educational support'},
    'Outreach':    {'units': total_visitations,  'label': "home visitation to a girl's family"},
    'Maintenance': {'units': total_bed_months,   'label': 'month of safehouse upkeep'},
    'Transport':   {'units': total_residents,    'label': 'transport trip for a resident'},
}

impact_rates = []

for area, allocated in total_by_area.items():
    mapping = PROGRAM_MAPPING.get(area)
    if mapping is None:
        print(f"  ⚠ Unknown program area '{area}' — skipping")
        continue
    if mapping['units'] == 0 or allocated <= 0:
        print(f"  ⚠ {area}: zero units ({mapping['units']}) or zero allocation (₱{allocated:,.2f}) — skipping")
        continue
    
    cost = allocated / mapping['units']
    impact_rates.append({
        'impact_category': area,
        'cost_per_unit':   round(cost, 2),
        'unit_label':      mapping['label'],
        'total_allocated': round(allocated, 2),
        'total_units':     round(mapping['units'], 2),
    })
    print(f"  ✓ {area}: ₱{cost:,.2f} per {mapping['label']}")

rates_df = pd.DataFrame(impact_rates)
print(f"\n{len(rates_df)} impact rates computed.")
rates_df

## 4b. Ratio Integrity Validation

Before using these ratios on the donate page, we verify:
1. **Data completeness** — no program areas silently dropped
2. **Plausibility bounds** — ratios fall within domain-reasonable ranges for Philippine nonprofits
3. **Temporal stability** — ratios don't swing wildly quarter-to-quarter (CV < 0.3 = safe to display)

In [ ]:
# --- Cell A: Data completeness checks ---
print("=== Data Completeness Checks ===\n")

# 1. Check all program areas in data are mapped
data_areas = set(allocations['program_area'].unique())
mapped_areas = set(PROGRAM_MAPPING.keys())
unmapped = data_areas - mapped_areas
unused_mappings = mapped_areas - data_areas

if unmapped:
    print(f"  WARNING: {len(unmapped)} program area(s) in data but NOT mapped: {unmapped}")
else:
    print(f"  PASS: All {len(data_areas)} program areas in data are mapped")

if unused_mappings:
    print(f"  INFO: {len(unused_mappings)} mapping(s) have no data: {unused_mappings}")

# 2. Check NaN coercion rate on amount_allocated
original_count = len(allocations) + allocations['amount_allocated'].isna().sum()
nan_rate = allocations['amount_allocated'].isna().sum() / max(original_count, 1) * 100
print(f"\n  Amount NaN coercion rate: {nan_rate:.1f}%", "PASS" if nan_rate < 5 else "WARNING (>5%)")

# 3. Cross-check: total allocations vs total monetary donations
donations['amount_numeric'] = pd.to_numeric(donations.get('amount', pd.Series(dtype='float64')), errors='coerce')
monetary_total = donations.loc[donations.get('donation_type', pd.Series()) == 'Monetary', 'amount_numeric'].sum()
alloc_total = allocations['amount_allocated'].sum()
gap_pct = abs(alloc_total - monetary_total) / max(monetary_total, 1) * 100
print(f"\n  Monetary donations total: ₱{monetary_total:,.2f}")
print(f"  Allocations total:       ₱{alloc_total:,.2f}")
print(f"  Gap: {gap_pct:.1f}%", "— PASS" if gap_pct < 20 else "— INFO: some donations may lack allocations")

In [ ]:
# --- Cell B: Ratio plausibility bounds ---
# Domain-informed bounds for Philippine nonprofit operations (PHP per unit)
PLAUSIBILITY_BOUNDS = {
    'Operations':  (500, 10000),
    'Wellbeing':   (100, 5000),
    'Education':   (200, 8000),
    'Outreach':    (100, 5000),
    'Maintenance': (200, 8000),
    'Transport':   (50, 5000),
}

print("=== Ratio Plausibility Bounds ===\n")
print(f"  {'Category':<14} {'Cost/Unit':>10} {'Lower':>8} {'Upper':>8}  Status")
print(f"  {'─'*14} {'─'*10} {'─'*8} {'─'*8}  {'─'*8}")

all_plausible = True
for _, row in rates_df.iterrows():
    cat = row['impact_category']
    cost = row['cost_per_unit']
    lo, hi = PLAUSIBILITY_BOUNDS.get(cat, (0, float('inf')))
    ok = lo <= cost <= hi
    if not ok:
        all_plausible = False
    status = "PASS" if ok else "FAIL"
    print(f"  {cat:<14} ₱{cost:>8,.2f} ₱{lo:>6,} ₱{hi:>6,}  {status}")

print(f"\n  Overall: {'ALL PASS' if all_plausible else 'SOME OUT OF BOUNDS — review data'}")

In [ ]:
# --- Cell C: Temporal stability ---
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("=== Temporal Stability Analysis ===\n")

# Parse dates and assign half-year periods (quarters may be too sparse with 521 rows)
alloc_dated = allocations.dropna(subset=['allocation_date']).copy()
alloc_dated['half_year'] = alloc_dated['allocation_date'].dt.to_period('6M')

# Parse metrics dates for denominator computation per period
metrics_copy = metrics.copy()
metrics_copy['month_start'] = pd.to_datetime(metrics_copy.get('month_start', pd.Series(dtype='str')), errors='coerce')
metrics_copy['half_year'] = metrics_copy['month_start'].dt.to_period('6M')

recordings_copy = recordings.copy()
recordings_copy['session_date'] = pd.to_datetime(recordings_copy.get('session_date', pd.Series(dtype='str')), errors='coerce')
recordings_copy['half_year'] = recordings_copy['session_date'].dt.to_period('6M')

visitations_copy = visitations.copy()
visitations_copy['visit_date'] = pd.to_datetime(visitations_copy.get('visit_date', pd.Series(dtype='str')), errors='coerce')
visitations_copy['half_year'] = visitations_copy['visit_date'].dt.to_period('6M')

education_copy = education.copy()
education_copy['record_date'] = pd.to_datetime(education_copy.get('record_date', pd.Series(dtype='str')), errors='coerce')
education_copy['half_year'] = education_copy['record_date'].dt.to_period('6M')

# Compute ratios per half-year for key program areas
periods = sorted(alloc_dated['half_year'].dropna().unique())
temporal_rows = []

for period in periods:
    period_alloc = alloc_dated[alloc_dated['half_year'] == period]
    period_by_area = period_alloc.groupby('program_area')['amount_allocated'].sum()
    
    # Compute denominators for this period
    period_metrics = metrics_copy[metrics_copy['half_year'] == period]
    period_bed_months = period_metrics['active_residents'].sum() if len(period_metrics) > 0 else 0
    period_sessions = len(recordings_copy[recordings_copy['half_year'] == period])
    period_visits = len(visitations_copy[visitations_copy['half_year'] == period])
    period_edu = len(education_copy[education_copy['half_year'] == period])
    
    denom_map = {
        'Operations': period_bed_months,
        'Wellbeing': period_sessions,
        'Education': period_edu,
        'Outreach': period_visits,
        'Maintenance': period_bed_months,
    }
    
    for area, alloc_amt in period_by_area.items():
        denom = denom_map.get(area, 0)
        if denom > 0 and alloc_amt > 0:
            temporal_rows.append({
                'period': str(period),
                'program_area': area,
                'cost_per_unit': alloc_amt / denom,
            })

temporal_df = pd.DataFrame(temporal_rows)

if len(temporal_df) > 0:
    # Compute CV per program area
    cv_df = temporal_df.groupby('program_area')['cost_per_unit'].agg(['mean', 'std', 'count'])
    cv_df['cv'] = cv_df['std'] / cv_df['mean']
    cv_df['stability'] = cv_df['cv'].apply(lambda x: 'STABLE' if x < 0.3 else ('MODERATE' if x < 0.5 else 'VOLATILE'))
    
    print(f"  {'Program Area':<14} {'Mean':>10} {'Std':>10} {'CV':>6}  {'Periods':>7}  Status")
    print(f"  {'─'*14} {'─'*10} {'─'*10} {'─'*6}  {'─'*7}  {'─'*8}")
    for area, row in cv_df.iterrows():
        print(f"  {area:<14} ₱{row['mean']:>8,.2f} ₱{row['std']:>8,.2f} {row['cv']:>5.2f}  {int(row['count']):>7}  {row['stability']}")
    
    # Plot temporal trends
    fig, ax = plt.subplots(figsize=(10, 5))
    for area in temporal_df['program_area'].unique():
        area_data = temporal_df[temporal_df['program_area'] == area].sort_values('period')
        ax.plot(area_data['period'], area_data['cost_per_unit'], marker='o', label=area)
    ax.set_xlabel('Half-Year Period')
    ax.set_ylabel('Cost per Unit (PHP)')
    ax.set_title('Temporal Stability: Cost-per-Unit Ratios Over Time')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('temporal_stability.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("\n  (Plot saved to temporal_stability.png)")
else:
    print("  Not enough dated allocation data for temporal analysis.")

## 5. Explanatory Analysis — What Drives Cost-per-Unit Variation?

Using OLS regression at the safehouse-month level to understand which factors are
associated with higher or lower cost-per-unit. This is an **association analysis**,
not causal inference — allocation rules may be administrative rather than cost-reflective.

Follows the pattern of `social-media-effectiveness.ipynb` (statsmodels OLS with HC3 robust SEs).

In [ ]:
# --- Cell D: Build safehouse-month panel ---
print("=== Building Safehouse-Month Panel ===\n")

# Parse metrics dates
metrics_panel = metrics.copy()
metrics_panel['month_start'] = pd.to_datetime(metrics_panel['month_start'], errors='coerce')
metrics_panel = metrics_panel.dropna(subset=['month_start', 'safehouse_id'])
metrics_panel['year_month'] = metrics_panel['month_start'].dt.to_period('M')

# Aggregate allocations to safehouse-month level (focus on Operations = shelter)
alloc_panel = allocations.dropna(subset=['allocation_date', 'safehouse_id']).copy()
alloc_panel['year_month'] = alloc_panel['allocation_date'].dt.to_period('M')

# Sum allocations per (safehouse, month, program_area)
alloc_monthly = alloc_panel.groupby(['safehouse_id', 'year_month', 'program_area'])['amount_allocated'].sum().reset_index()

# Focus on Operations (shelter) for OLS — largest program area with clear denominator
ops_alloc = alloc_monthly[alloc_monthly['program_area'] == 'Operations'].copy()
ops_alloc = ops_alloc.rename(columns={'amount_allocated': 'ops_allocated'})

# Join to metrics panel
panel = metrics_panel.merge(
    ops_alloc[['safehouse_id', 'year_month', 'ops_allocated']],
    on=['safehouse_id', 'year_month'],
    how='inner'
)

# Compute cost-per-bed-month at the safehouse-month level
panel['cost_per_bed_month'] = panel['ops_allocated'] / panel['active_residents']
panel = panel.replace([np.inf, -np.inf], np.nan).dropna(subset=['cost_per_bed_month'])

# Numeric features from metrics
for col in ['process_recording_count', 'home_visitation_count', 'incident_count',
            'avg_education_progress', 'avg_health_score']:
    panel[col] = pd.to_numeric(panel.get(col, pd.Series(dtype='float64')), errors='coerce').fillna(0)

# Add quarter feature
panel['quarter'] = panel['month_start'].dt.quarter

# Join safehouse region
if 'region' in safehouses.columns and 'safehouse_id' in safehouses.columns:
    panel = panel.merge(safehouses[['safehouse_id', 'region']], on='safehouse_id', how='left')
else:
    panel['region'] = 'Unknown'

print(f"  Panel rows: {len(panel)}")
print(f"  Safehouses: {panel['safehouse_id'].nunique()}")
print(f"  Date range: {panel['month_start'].min().date()} to {panel['month_start'].max().date()}")
print(f"\n  Target variable (cost_per_bed_month) summary:")
print(panel['cost_per_bed_month'].describe().to_string())

In [ ]:
# --- Cell E: OLS regression + interpretation ---
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

print("=== OLS Regression: Drivers of Cost-per-Bed-Month ===\n")

# Prepare features
feature_cols = ['active_residents', 'process_recording_count', 'home_visitation_count',
                'incident_count', 'quarter']

# Add region dummies if multiple regions exist
if panel['region'].nunique() > 1:
    region_dummies = pd.get_dummies(panel['region'], prefix='region', drop_first=True, dtype=float)
    ols_data = pd.concat([panel[feature_cols + ['cost_per_bed_month']], region_dummies], axis=1)
    feature_cols = feature_cols + list(region_dummies.columns)
else:
    ols_data = panel[feature_cols + ['cost_per_bed_month']].copy()

ols_data = ols_data.dropna()

X = sm.add_constant(ols_data[feature_cols].astype(float))
y = ols_data['cost_per_bed_month']

# Fit OLS with HC3 robust standard errors
model = sm.OLS(y, X).fit(cov_type='HC3')
print(model.summary())

# VIF for multicollinearity
print("\n--- Variance Inflation Factors ---")
vif_data = pd.DataFrame({
    'Feature': X.columns[1:],  # skip constant
    'VIF': [variance_inflation_factor(X.values, i+1) for i in range(len(X.columns)-1)]
})
print(vif_data.to_string(index=False))
high_vif = vif_data[vif_data['VIF'] > 10]
if len(high_vif) > 0:
    print(f"\n  WARNING: {len(high_vif)} feature(s) with VIF > 10 (multicollinearity concern)")
else:
    print("\n  PASS: No features with VIF > 10")

# Interpretation
print("\n--- Coefficient Interpretation ---")
for feat in feature_cols:
    coef = model.params[feat]
    pval = model.pvalues[feat]
    sig = "***" if pval < 0.01 else ("**" if pval < 0.05 else ("*" if pval < 0.1 else ""))
    print(f"  {feat}: {coef:+.2f} PHP per unit change {sig}")

print(f"\n  R-squared: {model.rsquared:.3f}")
print(f"  Adj. R-squared: {model.rsquared_adj:.3f}")
print(f"\n  NOTE: These are associations, not causal effects. The allocation process")
print(f"  and occupancy may both be driven by unmeasured safehouse characteristics.")

In [ ]:
# --- Cell F: OLS diagnostics ---
from scipy import stats as sp_stats
import seaborn as sns

print("=== OLS Diagnostics ===\n")

residuals = model.resid
fitted = model.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Residuals vs Fitted
axes[0].scatter(fitted, residuals, alpha=0.5, s=20)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

# 2. Q-Q Plot
sp_stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('Normal Q-Q Plot')

# 3. Residual histogram
axes[2].hist(residuals, bins=25, edgecolor='black', alpha=0.7)
axes[2].set_xlabel('Residual')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Residual Distribution')

plt.tight_layout()
plt.savefig('ols_diagnostics.png', dpi=100, bbox_inches='tight')
plt.show()

# Breusch-Pagan test for heteroscedasticity
from statsmodels.stats.diagnostic import het_breuschpagan
bp_stat, bp_pval, _, _ = het_breuschpagan(residuals, X)
print(f"  Breusch-Pagan test: stat={bp_stat:.2f}, p-value={bp_pval:.4f}")
print(f"  {'PASS: No evidence of heteroscedasticity (p > 0.05)' if bp_pval > 0.05 else 'NOTE: Heteroscedasticity detected — HC3 robust SEs already in use'}")

# Jarque-Bera normality test
jb_stat, jb_pval = sp_stats.jarque_bera(residuals)
print(f"\n  Jarque-Bera test: stat={jb_stat:.2f}, p-value={jb_pval:.4f}")
print(f"  {'PASS: Residuals are approximately normal' if jb_pval > 0.05 else 'NOTE: Non-normal residuals — OLS estimates still unbiased but CIs may be approximate'}")

## 6. Predictive Model — Donation Impact "Zestimate"

**Analogy:** Zillow pre-computes property value estimates from historical sales data so users
see instant predictions. We do the same — pre-compute impact ratios from historical data so donors
see instant impact predictions. The accuracy question is: **"How well do our stored ratios predict
actual impact delivery?"**

**Approach:** Rolling temporal backtest — compute ratios using only past data, then measure how
accurately they predict the actual cost-per-unit in future periods.

In [ ]:
# --- Cell G: Rolling backtest — predict impact from historical ratios ---
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

print("=== Donation Impact Prediction: Rolling Backtest ===\n")

# Build a dataset of (donation, program_area, period) with actual cost-per-unit
# For each donation allocation, we know the amount and program area.
# We'll predict: impact_units = floor(amount / predicted_ratio)
# and compare to: actual_units = floor(amount / actual_period_ratio)

# Use the temporal ratios computed earlier in Cell C
if len(temporal_df) > 0:
    # Get unique periods sorted chronologically
    all_periods = sorted(temporal_df['period'].unique())
    
    backtest_rows = []
    
    for i, test_period in enumerate(all_periods):
        if i < 2:  # Need at least 2 prior periods for training
            continue
        
        train_periods = all_periods[:i]
        
        for area in temporal_df['program_area'].unique():
            # Train: compute global ratio from all prior periods
            train_data = temporal_df[
                (temporal_df['period'].isin(train_periods)) &
                (temporal_df['program_area'] == area)
            ]
            if len(train_data) == 0:
                continue
            
            predicted_ratio = train_data['cost_per_unit'].mean()  # Our "Zestimate"
            
            # Test: actual ratio for this period
            test_data = temporal_df[
                (temporal_df['period'] == test_period) &
                (temporal_df['program_area'] == area)
            ]
            if len(test_data) == 0:
                continue
            
            actual_ratio = test_data['cost_per_unit'].values[0]
            
            # Simulate a ₱5,000 donation
            test_amount = 5000
            predicted_units = int(test_amount // predicted_ratio)
            actual_units = int(test_amount // actual_ratio)
            
            backtest_rows.append({
                'period': test_period,
                'program_area': area,
                'predicted_ratio': predicted_ratio,
                'actual_ratio': actual_ratio,
                'predicted_units': predicted_units,
                'actual_units': actual_units,
                'ratio_error': abs(predicted_ratio - actual_ratio),
                'unit_error': abs(predicted_units - actual_units),
            })
    
    backtest_df = pd.DataFrame(backtest_rows)
    
    if len(backtest_df) > 0:
        print(f"  Backtest observations: {len(backtest_df)}")
        print(f"  Program areas tested: {backtest_df['program_area'].nunique()}")
        print(f"  Periods tested: {backtest_df['period'].nunique()}\n")
        
        # Overall accuracy metrics
        ratio_mae = mean_absolute_error(backtest_df['actual_ratio'], backtest_df['predicted_ratio'])
        ratio_rmse = np.sqrt(mean_squared_error(backtest_df['actual_ratio'], backtest_df['predicted_ratio']))
        ratio_r2 = r2_score(backtest_df['actual_ratio'], backtest_df['predicted_ratio'])
        unit_mae = backtest_df['unit_error'].mean()
        
        print("  --- Baseline Model (Global Average Ratio) ---")
        print(f"  Ratio MAE:  ₱{ratio_mae:,.2f}")
        print(f"  Ratio RMSE: ₱{ratio_rmse:,.2f}")
        print(f"  Ratio R²:   {ratio_r2:.3f}")
        print(f"  Unit MAE:   {unit_mae:.1f} impact units (for ₱5,000 donation)")
        
        # Per-area accuracy
        print(f"\n  --- Accuracy by Program Area (₱5,000 donation) ---")
        print(f"  {'Area':<14} {'Ratio MAE':>10} {'Unit MAE':>9} {'Pct Accurate':>13}")
        print(f"  {'─'*14} {'─'*10} {'─'*9} {'─'*13}")
        for area in backtest_df['program_area'].unique():
            area_data = backtest_df[backtest_df['program_area'] == area]
            area_ratio_mae = area_data['ratio_error'].mean()
            area_unit_mae = area_data['unit_error'].mean()
            pct_exact = (area_data['unit_error'] == 0).mean() * 100
            print(f"  {area:<14} ₱{area_ratio_mae:>8,.2f} {area_unit_mae:>8.1f} {pct_exact:>11.0f}%")
    else:
        print("  Not enough temporal data for backtest.")
else:
    print("  Temporal data not available — skipping backtest.")

In [ ]:
# --- Cell H: ML model comparison (Linear Regression vs Decision Tree) ---
print("=== ML Model Comparison: Can We Beat the Simple Ratio? ===\n")

if len(backtest_df) > 0 and len(panel) > 10:
    # Use the safehouse-month panel for ML prediction
    # Target: cost_per_bed_month  Features: occupancy, service counts, quarter, safehouse
    ml_panel = panel.copy()
    ml_panel = ml_panel.sort_values('month_start')
    
    # Encode safehouse_id as numeric
    le_sh = LabelEncoder()
    ml_panel['safehouse_encoded'] = le_sh.fit_transform(ml_panel['safehouse_id'].astype(str))
    
    ml_features = ['active_residents', 'process_recording_count', 'home_visitation_count',
                   'incident_count', 'quarter', 'safehouse_encoded']
    
    ml_X = ml_panel[ml_features].fillna(0)
    ml_y = ml_panel['cost_per_bed_month']
    
    # Temporal split: train on first 80%, test on last 20%
    split_idx = int(len(ml_panel) * 0.8)
    X_train, X_test = ml_X.iloc[:split_idx], ml_X.iloc[split_idx:]
    y_train, y_test = ml_y.iloc[:split_idx], ml_y.iloc[split_idx:]
    
    print(f"  Train: {len(X_train)} rows | Test: {len(X_test)} rows\n")
    
    # Naive baseline: predict the training mean
    baseline_pred = np.full(len(y_test), y_train.mean())
    baseline_mae = mean_absolute_error(y_test, baseline_pred)
    baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
    baseline_r2 = r2_score(y_test, baseline_pred)
    
    # Linear Regression
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    lr_pred = lr.predict(X_test)
    lr_mae = mean_absolute_error(y_test, lr_pred)
    lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
    lr_r2 = r2_score(y_test, lr_pred)
    
    # Decision Tree (with limited depth to prevent overfitting)
    dt = DecisionTreeRegressor(max_depth=4, random_state=42)
    dt.fit(X_train, y_train)
    dt_pred = dt.predict(X_test)
    dt_mae = mean_absolute_error(y_test, dt_pred)
    dt_rmse = np.sqrt(mean_squared_error(y_test, dt_pred))
    dt_r2 = r2_score(y_test, dt_pred)
    
    print(f"  {'Model':<25} {'MAE (PHP)':>10} {'RMSE (PHP)':>11} {'R²':>7}")
    print(f"  {'─'*25} {'─'*10} {'─'*11} {'─'*7}")
    print(f"  {'Naive Baseline (mean)':<25} ₱{baseline_mae:>8,.2f} ₱{baseline_rmse:>9,.2f} {baseline_r2:>6.3f}")
    print(f"  {'Linear Regression':<25} ₱{lr_mae:>8,.2f} ₱{lr_rmse:>9,.2f} {lr_r2:>6.3f}")
    print(f"  {'Decision Tree (d=4)':<25} ₱{dt_mae:>8,.2f} ₱{dt_rmse:>9,.2f} {dt_r2:>6.3f}")
    
    # Determine winner
    best_model = 'Linear Regression' if lr_mae < dt_mae else 'Decision Tree'
    best_mae = min(lr_mae, dt_mae)
    beats_baseline = best_mae < baseline_mae
    
    print(f"\n  Best ML model: {best_model} (MAE: ₱{best_mae:,.2f})")
    if beats_baseline:
        improvement = (1 - best_mae / baseline_mae) * 100
        print(f"  ML beats baseline by {improvement:.1f}%")
    else:
        print(f"  Baseline is competitive — simple ratio approach is adequate")
    
    # Translate to donor-facing impact units
    test_donation = 5000
    mae_in_units = test_donation / (y_test.mean() - best_mae) - test_donation / y_test.mean()
    print(f"\n  For a ₱{test_donation:,} donation, prediction error ~{abs(mae_in_units):.1f} impact units")
    
    # Plot: predicted vs actual
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].scatter(y_test, lr_pred, alpha=0.5, label='Linear Regression')
    axes[0].scatter(y_test, dt_pred, alpha=0.5, label='Decision Tree', marker='x')
    min_val = min(y_test.min(), min(lr_pred.min(), dt_pred.min()))
    max_val = max(y_test.max(), max(lr_pred.max(), dt_pred.max()))
    axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect prediction')
    axes[0].set_xlabel('Actual Cost-per-Unit (PHP)')
    axes[0].set_ylabel('Predicted Cost-per-Unit (PHP)')
    axes[0].set_title('Predicted vs Actual')
    axes[0].legend()
    
    # Residual distribution for best model
    best_resid = (y_test - lr_pred) if lr_mae < dt_mae else (y_test - dt_pred)
    axes[1].hist(best_resid, bins=20, edgecolor='black', alpha=0.7)
    axes[1].axvline(0, color='red', linestyle='--')
    axes[1].set_xlabel('Prediction Error (PHP)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title(f'{best_model} — Residual Distribution')
    
    plt.tight_layout()
    plt.savefig('prediction_accuracy.png', dpi=100, bbox_inches='tight')
    plt.show()
else:
    print("  Not enough data for ML model comparison.")

## 7. Summary of Relationships and Causal Claims

### Key Findings

**Cost Efficiency by Program Area:**
- The ratios computed in Section 4 show how much each program area costs per unit of impact.
  Lower cost-per-unit means higher cost-efficiency (more impact per peso donated).
- Operations (shelter) and Wellbeing (counseling) are the most visible to donors and tend
  to dominate the impact cards shown on the donate page.

**What Drives Cost Variation (OLS, Section 5):**
- The OLS model identifies which safehouse-level and temporal factors are associated with
  higher or lower shelter cost-per-bed-month.
- **Active residents** (occupancy) is expected to show a negative association — higher occupancy
  spreads fixed costs over more girls, reducing per-unit cost (economies of scale).
- **Season (quarter)** may capture giving patterns — end-of-year donations tend to spike,
  but operational costs remain steady, affecting the ratio.

**Causal Limitations:**
- These are **associations, not causal effects**. We cannot say "increasing occupancy will
  reduce costs" because both occupancy and cost allocation may be driven by unmeasured factors
  (e.g., safehouse size, management quality, regional cost of living).
- The allocation process is partly administrative — program areas may receive funds based on
  budgets rather than actual cost, which limits how literally we can interpret the ratios.
- With synthetic data, the relationships reflect the data generation process rather than
  true Philippine nonprofit economics.

**Predictive Accuracy (Section 6):**
- The rolling backtest and ML comparison test whether our stored ratios accurately predict
  future impact delivery — the same question Zillow asks about their Zestimates.
- If the naive baseline (global average) performs comparably to ML models, it validates our
  current approach: simple ratio computation is sufficient for donor-facing display.
- The temporal stability analysis (Section 4b) further confirms whether ratios are stable
  enough to trust for real-time display.

### Implications for the Donate Page
- Impact cards should only display program areas with **stable ratios** (CV < 0.3) and
  **plausible bounds** (within domain expectations).
- The displayed numbers use `Math.floor()` to ensure whole, meaningful impact counts.
- If a program area's ratio becomes volatile, the frontend falls back to hardcoded sample
  rates (defined in `DonationImpactCards.tsx`) until the pipeline stabilizes.

## 8. Sanity Check — What Would a ₱5,000 Donation Look Like?

In [ ]:
test_amount = 5000
print(f"A ₱{test_amount:,} donation could provide:\n")

for _, row in rates_df.iterrows():
    units = int(test_amount // row['cost_per_unit'])
    if units >= 1:
        print(f"  • {units} {row['unit_label']}")
    else:
        print(f"  • (less than 1 {row['unit_label']} — would not display)")

## 9. Write to Database

In [ ]:
if USE_DB and len(rates_df) > 0:
    _model_version = _dt.date.today().isoformat()
    _now = _dt.datetime.utcnow()

    rates_df['scored_at'] = _now
    rates_df['model_version'] = _model_version
    rates_df['is_current'] = 1

    with engine.begin() as _conn:
        # Mark previous rates as non-current
        _conn.execute(text("UPDATE ml_donation_impact_rates SET is_current = 0 WHERE is_current = 1"))

        # Insert new rates
        rates_df[['impact_category', 'cost_per_unit', 'unit_label',
                   'total_allocated', 'total_units', 'scored_at',
                   'model_version', 'is_current']].to_sql(
            'ml_donation_impact_rates', _conn, if_exists='append', index=False
        )

    print(f"✓ Wrote {len(rates_df)} impact rates to ml_donation_impact_rates (model v{_model_version})")
else:
    print("Skipping DB write — USE_DB is False (no DB_CONNECTION_STRING set)")
    if len(rates_df) == 0:
        print("No rates to write.")